In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

cifar_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

print(f"CIFAR-10 loaded: {len(cifar_data)} images")
print(f"Image shape: {cifar_data[0][0].shape}")  # Should be (3, 32, 32)

In [ ]:
class ColorizationDataset(Dataset):
    """
    A dataset for image colorization.
    Returns (grayscale_image, color_image) pairs.

    Args:
        cifar_dataset: The CIFAR-10 dataset (already transformed to tensors)
    """

    def __init__(self, cifar_dataset):
        self.dataset = cifar_dataset

    def __len__(self):
        return len(self.dataset)

    def rgb_to_grayscale(self, img):
        r = img[0]
        g = img[1]
        b = img[2]

        gray = 0.299 * r + 0.587 * g + 0.114 * b
        gray = gray.unsqueeze(0)

        return gray

    def __getitem__(self, idx):
        color_image, _ = self.dataset[idx]
        grayscale_image = self.rgb_to_grayscale(color_image)

        return grayscale_image, color_image


In [ ]:
colorization_dataset = ColorizationDataset(cifar_data)

gray_img, color_img = colorization_dataset[0]

print("Grayscale image shape:", gray_img.shape)
print("Color image shape:", color_img.shape)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(6, 3))

axes[0].imshow(gray_img[0], cmap="gray")
axes[0].set_title("Grayscale (Input)")
axes[0].axis("off")

axes[1].imshow(color_img.permute(1, 2, 0))
axes[1].set_title("Color (Target)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
dataloader = DataLoader(
    colorization_dataset,
    batch_size=8,
    shuffle=True
)

gray_batch, color_batch = next(iter(dataloader))

print("Batch grayscale shape:", gray_batch.shape)
print("Batch color shape:", color_batch.shape)
